In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("RetailSalesAnalysis") \
    .master("local[*]") \
    .getOrCreate()

print("✅ Spark Started")

✅ Spark Started


In [4]:
df = spark.read.csv("retail_sales_data_BDT.csv", header=True, inferSchema=True)

df.show(5)

+--------------+----------+----------+------------+-------------+----------+--------+-----------+-----------+--------+---------+--------------+
|Transaction_ID|Order_Date|Product_ID|Product_Name|     Category|Unit_Price|Quantity|Total_Sales|Customer_ID|Store_ID|     City|Payment_Method|
+--------------+----------+----------+------------+-------------+----------+--------+-----------+-----------+--------+---------+--------------+
|         T1000|20-05-2024|     P1002| Wheat Flour|      Grocery|       399|       1|        399|       C350|     S02|   Mumbai|           UPI|
|         T1001|29-10-2024|     P1002| Wheat Flour|      Grocery|        64|       9|        576|       C532|     S01|Hyderabad|          Cash|
|         T1002|04-11-2024|     P2001|     Shampoo|Personal Care|       278|       4|       1112|       C127|     S02|  Kolkata|           UPI|
|         T1003|17-08-2024|     P4002|      Butter|        Dairy|       132|       7|        924|       C703|     S03|Hyderabad|        

In [5]:
from pyspark.sql.functions import col, month, to_date

# First check original data
df.select("Order_Date").show(5)

# Trim spaces (VERY IMPORTANT)
from pyspark.sql.functions import trim
df = df.withColumn("Order_Date", trim(col("Order_Date")))

# Try multiple formats safely
df = df.withColumn("Order_Date",
    to_date(col("Order_Date"), "dd-MM-yyyy")
)

# Check if conversion worked
df.select("Order_Date").show(5)

# Create Month column
df = df.withColumn("Month", month(col("Order_Date")))

# Calculate Total Sales
df = df.withColumn("Total_Sales", col("Unit_Price") * col("Quantity"))

df.show(5)

+----------+
|Order_Date|
+----------+
|20-05-2024|
|29-10-2024|
|04-11-2024|
|17-08-2024|
|20-03-2024|
+----------+
only showing top 5 rows
+----------+
|Order_Date|
+----------+
|2024-05-20|
|2024-10-29|
|2024-11-04|
|2024-08-17|
|2024-03-20|
+----------+
only showing top 5 rows
+--------------+----------+----------+------------+-------------+----------+--------+-----------+-----------+--------+---------+--------------+-----+
|Transaction_ID|Order_Date|Product_ID|Product_Name|     Category|Unit_Price|Quantity|Total_Sales|Customer_ID|Store_ID|     City|Payment_Method|Month|
+--------------+----------+----------+------------+-------------+----------+--------+-----------+-----------+--------+---------+--------------+-----+
|         T1000|2024-05-20|     P1002| Wheat Flour|      Grocery|       399|       1|        399|       C350|     S02|   Mumbai|           UPI|    5|
|         T1001|2024-10-29|     P1002| Wheat Flour|      Grocery|        64|       9|        576|       C532|     S01|

In [6]:
from pyspark.sql.functions import sum as spark_sum

df.select(spark_sum("Total_Sales")).show()

df.groupBy("Product_Name").sum("Total_Sales").show()

df.groupBy("City").sum("Total_Sales").show()

df.groupBy("Month").sum("Total_Sales").show()

df.groupBy("Payment_Method").sum("Total_Sales").show()

+----------------+
|sum(Total_Sales)|
+----------------+
|         2870124|
+----------------+

+------------+----------------+
|Product_Name|sum(Total_Sales)|
+------------+----------------+
|      Butter|          278849|
|        Soap|          279366|
|        Milk|          274160|
| Cooking Oil|          257023|
|       Chips|          323002|
|        Rice|          264776|
|  Soft Drink|          280550|
|    Biscuits|          318413|
| Wheat Flour|          321644|
|     Shampoo|          272341|
+------------+----------------+

+---------+----------------+
|     City|sum(Total_Sales)|
+---------+----------------+
|Bangalore|          507120|
|  Chennai|          466827|
|   Mumbai|          471200|
|  Kolkata|          484672|
|    Delhi|          508547|
|Hyderabad|          431758|
+---------+----------------+

+-----+----------------+
|Month|sum(Total_Sales)|
+-----+----------------+
|   12|          258079|
|    1|          238682|
|    6|          249445|
|    3|       

In [7]:
df.toPandas().to_csv("cleaned_sales_spark.csv", index=False)

print("✅ File saved successfully")

✅ File saved successfully
